<a href="https://colab.research.google.com/github/Saliyah-53/Saliyah-53/blob/main/StanceEval2026_Baseline_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **StanceEval2026**
### **Arabic Stance Detection Shared Task Baseline**


This notebook provides a clean and reproducible baseline for the
StanceEval2026 shared task on target-specific Arabic stance detection
using AraBERT-twitter.

---

## **Baseline Configuration**
This baseline is provided as a simple reference implementation.
Participants may adapt and extend any component of the pipeline,
including preprocessing, training strategy, and model selection.

| Setting | Value |
|---|---|
| Model | `aubmindlab/bert-base-arabertv02-twitter` |
| Input format | `(target, text)` |
| Max length | `128` |
| Batch size | `32` |
| Epochs | `20` |
| Learning rate | `2e-5` |
| Best checkpoint | highest dev `Favg2` |
| Labels | `Against`, `Favor`, `None` |

---

## **Features**

- Arabic preprocessing
- Dev evaluation using:
  - `Favg2`
  - `Favg3`
- Submission file generation for:
  - `test_seen.csv`
  - `test_unseen.csv`

---

## **Output Files**

- `submission_seen.csv`
- `submission_unseen.csv`

This baseline is intended to provide participants with a simple
and reproducible starting point for experimentation.

---
## **Expected Data Structure**

The notebook expects the following files inside the `data/` directory:

- `train.csv`
- `dev.csv`
- `test_seen.csv`
- `test_unseen.csv`

## **Requirements**

Recommended environment:

- Python 3.10+
- PyTorch 2.x
- Transformers 4.x

Main packages:
- transformers
- torch
- pandas
- scikit-learn
- tqdm
- sentencepiece

In [2]:
# ============================================================
# 1) Install Required Packages
# ============================================================
# Skip this cell if packages are already installed
#!pip install -q transformers scikit-learn pandas tqdm sentencepiece

In [3]:
# ============================================================
# 2) Optional: Mount Google Drive (Colab)
# ============================================================

# Run this cell only if you use Google Drive in Colab

#from google.colab import drive
#drive.mount('/content/drive')

In [4]:
# ============================================================
# 3) Imports
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm.auto import tqdm

In [5]:
# ============================================================
# 4) Configuration
# ============================================================

# =========================
# PATHS
# =========================
# Expected structure:
# project/
# ├── baseline.ipynb
# └── data/
#     ├── train.csv
#     ├── dev.csv
#     ├── test_seen.csv
#     └── test_unseen.csv

BASE_DIR = "."

# If using Google Drive, comment the line above and use this instead:
# BASE_DIR = "/content/drive/MyDrive/StanceEval2026"

DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
DEV_PATH = f"{DATA_DIR}/dev.csv"
TEST_SEEN_PATH = f"{DATA_DIR}/test_seen.csv"
TEST_UNSEEN_PATH = f"{DATA_DIR}/test_unseen.csv"

OUTPUT_DIR = f"{BASE_DIR}/baseline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# COLUMNS
# =========================
TEXT_COL = "text"
TARGET_COL = "target"
LABEL_COL = "stance"
ID_COL = "id"

# =========================
# MODEL SETTINGS
# =========================
MODEL_DISPLAY_NAME = "AraBERTv0.2-Twitter"
MODEL_HF_NAME = "aubmindlab/bert-base-arabertv02-twitter"

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-5
SEED = 42

LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "None": 2,
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

required_files = [TRAIN_PATH, DEV_PATH]
for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File not found: {path}\n"
            "Please make sure all files are inside the data/ folder."
        )

print("All dataset files found.")
print("Model:", MODEL_DISPLAY_NAME)
print("Output directory:", OUTPUT_DIR)

All dataset files found.
Model: AraBERTv0.2-Twitter
Output directory: ./baseline_outputs


In [6]:
# ============================================================
# 5) Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [7]:
# ============================================================
# 6) Arabic Text Preprocessing
# ============================================================

ARABIC_DIACRITICS = re.compile(r"ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ")
NON_ARABIC = re.compile(r"[^\u0600-\u06FF0-9\s]+")
MULTI_SPACE = re.compile(r"\s+")
REPEATED_CHAR = re.compile(r"(.)\1{2,}")

def preprocess_text(text):
    text = str(text)
    text = re.sub(ARABIC_DIACRITICS, "", text)
    text = re.sub(NON_ARABIC, " ", text)
    text = re.sub(REPEATED_CHAR, r"\1\1", text)
    text = re.sub(MULTI_SPACE, " ", text).strip()
    return text

In [8]:
# ============================================================
# 7) Load Labeled Data (Train / Dev)
# ============================================================

def load_labeled_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)

    needed = [TEXT_COL, TARGET_COL, LABEL_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    for c in needed:
        df[c] = df[c].astype(str).str.strip()

    df = df[
        (df[TEXT_COL] != "") &
        (df[TARGET_COL] != "") &
        (df[LABEL_COL] != "")
    ].copy()

    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)

    unknown_labels = sorted(set(df[LABEL_COL]) - set(LABEL2ID.keys()))
    if unknown_labels:
        raise ValueError(f"Unknown stance labels: {unknown_labels}")

    df["label"] = df[LABEL_COL].map(LABEL2ID).astype(int)
    return df

# ============================================================
# 8) Load Unlabeled Test Data
# ============================================================
def load_unlabeled_test_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)

    needed = [ID_COL, TEXT_COL, TARGET_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    for c in needed:
        df[c] = df[c].astype(str).str.strip()

    df = df[
        (df[ID_COL] != "") &
        (df[TEXT_COL] != "") &
        (df[TARGET_COL] != "")
    ].copy()

    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)
    return df


train_df = load_labeled_dataframe(TRAIN_PATH)
dev_df = load_labeled_dataframe(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

print("\nTrain label counts:")
print(train_df[LABEL_COL].value_counts())

print("\nDev label counts:")
print(dev_df[LABEL_COL].value_counts())

Train shape: (3502, 15)
Dev shape: (619, 15)

Train label counts:
stance
Favor      2148
Against    1021
None        333
Name: count, dtype: int64

Dev label counts:
stance
Favor      380
Against    180
None        59
Name: count, dtype: int64


In [9]:

# ============================================================
# 9) Dataset
# ============================================================
class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            row[TARGET_COL],
            row[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}

        if not self.is_test:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)

        return item

In [10]:
# ============================================================
# 10) Evaluation Metrics
# ============================================================

def compute_metrics(y_true, y_pred):
    f_against = f1_score(y_true, y_pred, labels=[0], average="macro", zero_division=0)
    f_favor = f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    f_none = f1_score(y_true, y_pred, labels=[2], average="macro", zero_division=0)

    favg2 = (f_favor + f_against) / 2.0
    favg3 = (f_favor + f_against + f_none) / 3.0
    acc = accuracy_score(y_true, y_pred)

    return {
        "F_favor": f_favor,
        "F_against": f_against,
        "F_none": f_none,
        "Favg2": favg2,
        "Favg3": favg3,
        "Acc": acc,
    }


@torch.no_grad()
def predict_labeled_loader(model, loader):
    model.eval()
    preds = []
    labels = []

    for batch in loader:
        labels.extend(batch["labels"].cpu().numpy().tolist())
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        pred = torch.argmax(outputs.logits, dim=1)

        preds.extend(pred.cpu().numpy().tolist())

    return labels, preds


@torch.no_grad()
def predict_unlabeled_loader(model, loader):
    model.eval()
    preds = []

    for batch in tqdm(loader, desc="Predicting"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        pred = torch.argmax(outputs.logits, dim=1)

        preds.extend(pred.cpu().numpy().tolist())

    return preds


def build_per_target_results(eval_df, pred_ids, model_name):
    df = eval_df.copy()
    df["pred"] = pred_ids

    row = {"Model": model_name}

    for target in sorted(df[TARGET_COL].unique()):
        sub = df[df[TARGET_COL] == target]
        m = compute_metrics(sub["label"], sub["pred"])
        safe_target = target.replace(" ", "_").replace("/", "_")

        row[f"{safe_target}_Favg2"] = m["Favg2"] * 100
        row[f"{safe_target}_Favg3"] = m["Favg3"] * 100

    overall = compute_metrics(df["label"], df["pred"])

    row["F_favor"] = overall["F_favor"] * 100
    row["F_against"] = overall["F_against"] * 100
    row["F_none"] = overall["F_none"] * 100
    row["Overall_Favg2"] = overall["Favg2"] * 100
    row["Overall_Favg3"] = overall["Favg3"] * 100
    row["Acc"] = overall["Acc"] * 100

    return pd.DataFrame([row]).round(2), df

In [11]:
# ============================================================
# 11) Tokenizer + DataLoaders
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_NAME)

train_ds = StanceDataset(train_df, tokenizer)
dev_ds = StanceDataset(dev_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 12) Load Model
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_HF_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/751k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# ============================================================
# 13) Training + Dev Evaluation
# ============================================================
model_base_dir = f"{OUTPUT_DIR}/{MODEL_DISPLAY_NAME}"
best_favg2_dir = f"{model_base_dir}/best_favg2"

os.makedirs(best_favg2_dir, exist_ok=True)

history = []
best_dev_favg2 = -1.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    pbar = tqdm(train_loader, desc=f"{MODEL_DISPLAY_NAME} | Epoch {epoch+1}/{EPOCHS}")

    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_loss / max(len(train_loader), 1)

    dev_labels, dev_preds = predict_labeled_loader(model, dev_loader)
    dev_metrics = compute_metrics(dev_labels, dev_preds)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "dev_F_favor": dev_metrics["F_favor"] * 100,
        "dev_F_against": dev_metrics["F_against"] * 100,
        "dev_F_none": dev_metrics["F_none"] * 100,
        "dev_Favg2": dev_metrics["Favg2"] * 100,
        "dev_Favg3": dev_metrics["Favg3"] * 100,
        "dev_Acc": dev_metrics["Acc"] * 100,
    })

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"dev_Favg2={dev_metrics['Favg2']*100:.2f} | "
        f"dev_Favg3={dev_metrics['Favg3']*100:.2f} | "
        f"dev_acc={dev_metrics['Acc']*100:.2f}"
    )

    if dev_metrics["Favg2"] > best_dev_favg2:
        best_dev_favg2 = dev_metrics["Favg2"]
        model.save_pretrained(best_favg2_dir)
        tokenizer.save_pretrained(best_favg2_dir)
        print("Saved BEST_FAVG2 checkpoint.")

history_df = pd.DataFrame(history)
history_path = f"{model_base_dir}/training_history.csv"
history_df.to_csv(history_path, index=False)

print("\nTraining history saved to:", history_path)
display(history_df)

AraBERTv0.2-Twitter | Epoch 1/20:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.6535 | dev_Favg2=78.25 | dev_Favg3=57.07 | dev_acc=76.41


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved BEST_FAVG2 checkpoint.


AraBERTv0.2-Twitter | Epoch 2/20:   0%|          | 0/110 [00:00<?, ?it/s]

...

AraBERTv0.2-Twitter | Epoch 19/20:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 19 | train_loss=0.0128 | dev_Favg2=82.06 | dev_Favg3=68.41 | dev_acc=79.97


AraBERTv0.2-Twitter | Epoch 20/20:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 20 | train_loss=0.0039 | dev_Favg2=81.02 | dev_Favg3=67.58 | dev_acc=78.84

Training history saved to: ./baseline_outputs/AraBERTv0.2-Twitter/training_history.csv


,epoch,train_loss,dev_F_favor,dev_F_against,dev_F_none,dev_Favg2,dev_Favg3,dev_Acc
0,1,0.653513,83.854167,72.636816,14.705882,78.245491,57.065622,76.413570
1,2,0.437936,85.552408,75.662651,42.735043,80.607529,67.983367,78.190630
2,3,0.295087,88.274045,79.057592,39.175258,83.665818,68.835631,81.583199
3,4,0.182628,88.888889,76.487252,43.243243,82.688071,69.539795,81.260097
4,5,0.117382,88.348271,76.750700,46.000000,82.549486,70.366324,81.583199
5,6,0.069380,88.476821,77.101449,39.130435,82.789135,68.236235,79.806139
6,7,0.039879,88.390501,75.428571,38.461538,81.909536,67.426870,79.483037
7,8,0.026239,87.282463,78.371501,40.816327,82.826982,68.823430,80.775444
8,9,0.016230,87.418936,77.685950,44.230769,82.552443,69.778552,80.936995
9,10,0.014400,88.633461,77.714286,45.714286,83.173873,70.687344,81.906300


In [13]:
# ============================================================
# 14) Final Dev Evaluation
# ============================================================

best_tokenizer = AutoTokenizer.from_pretrained(best_favg2_dir)
best_model = AutoModelForSequenceClassification.from_pretrained(best_favg2_dir).to(DEVICE)

dev_ds_best = StanceDataset(dev_df, best_tokenizer)
dev_loader_best = DataLoader(dev_ds_best, batch_size=BATCH_SIZE, shuffle=False)

dev_labels, dev_preds = predict_labeled_loader(best_model, dev_loader_best)
dev_metrics = compute_metrics(dev_labels, dev_preds)

print("\nFINAL DEV RESULTS USING BEST_FAVG2 CHECKPOINT")
print(f"F_favor:   {dev_metrics['F_favor']*100:.2f}")
print(f"F_against: {dev_metrics['F_against']*100:.2f}")
print(f"F_none:    {dev_metrics['F_none']*100:.2f}")
print(f"Favg2:     {dev_metrics['Favg2']*100:.2f}")
print(f"Favg3:     {dev_metrics['Favg3']*100:.2f}")
print(f"Accuracy:  {dev_metrics['Acc']*100:.2f}")

print("\nDEV CLASSIFICATION REPORT")
print(
    classification_report(
        dev_labels,
        dev_preds,
        labels=[0, 1, 2],
        target_names=["Against", "Favor", "None"],
        digits=4,
        zero_division=0
    )
)

dev_table, dev_predictions_df = build_per_target_results(
    dev_df,
    dev_preds,
    f"{MODEL_DISPLAY_NAME}_best_favg2"
)

print("\nDEV RESULTS PER TARGET USING BEST_FAVG2 CHECKPOINT")
display(dev_table)

dev_table_path = f"{model_base_dir}/dev_results_per_target_best_favg2.csv"
dev_table.to_csv(dev_table_path, index=False)

dev_predictions_df["pred_id"] = dev_predictions_df["pred"]
dev_predictions_df["pred_stance"] = [ID2LABEL[p] for p in dev_predictions_df["pred"]]

dev_pred_path = f"{model_base_dir}/dev_predictions_best_favg2.csv"
dev_predictions_df.to_csv(dev_pred_path, index=False)

print("\nDev predictions saved to:", dev_pred_path)
print("Dev per-target results saved to:", dev_table_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


FINAL DEV RESULTS USING BEST_FAVG2 CHECKPOINT
F_favor:   88.27
F_against: 79.06
F_none:    39.18
Favg2:     83.67
Favg3:     68.84
Accuracy:  81.58

DEV CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Against     0.7475    0.8389    0.7906       180
       Favor     0.8839    0.8816    0.8827       380
        None     0.5000    0.3220    0.3918        59

    accuracy                         0.8158       619
   macro avg     0.7105    0.6808    0.6884       619
weighted avg     0.8077    0.8158    0.8091       619


DEV RESULTS PER TARGET USING BEST_FAVG2 CHECKPOINT


,Model,Covid_Vaccine_Favg2,Covid_Vaccine_Favg3,Digital_Transformation_Favg2,Digital_Transformation_Favg3,Women_empowerment_Favg2,Women_empowerment_Favg3,F_favor,F_against,F_none,Overall_Favg2,Overall_Favg3,Acc
0,AraBERTv0.2-Twitter_best_favg2,79.05,63.07,77.15,67.12,88.07,73.53,88.27,79.06,39.18,83.67,68.84,81.58



Dev predictions saved to: ./baseline_outputs/AraBERTv0.2-Twitter/dev_predictions_best_favg2.csv
Dev per-target results saved to: ./baseline_outputs/AraBERTv0.2-Twitter/dev_results_per_target_best_favg2.csv


In [14]:
# ============================================================
# 15) Submission Generation: Predict on Unlabeled Test Files
# ============================================================

def generate_submission(test_path, output_filename):
    test_df = load_unlabeled_test_dataframe(test_path)

    test_ds = StanceDataset(test_df, best_tokenizer, is_test=True)

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    pred_ids = predict_unlabeled_loader(best_model, test_loader)

    pred_labels = [ID2LABEL[p] for p in pred_ids]

    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        LABEL_COL: pred_labels
    })

    output_path = f"{model_base_dir}/{output_filename}"

    submission.to_csv(output_path, index=False)

    print("Saved submission to:", output_path)

    return submission

In [15]:
# ============================================================
# 16) Generate Seen and Unseen Submissions
# ============================================================

submission_seen = generate_submission(
    TEST_SEEN_PATH,
    "submission_seen.csv"
)

submission_unseen = generate_submission(
    TEST_UNSEEN_PATH,
    "submission_unseen.csv"
)

FileNotFoundError: [Errno 2] No such file or directory: './data/test_seen.csv'

## **Notes**

- This baseline is intended as a simple and reproducible reference implementation for the shared task.
- Participants are encouraged to experiment with different preprocessing methods, training strategies, and model architectures.
- Performance may vary depending on hardware, random initialization, and software environment.